In [1]:
import pandas as pd
import random
from datetime import datetime, timedelta

# Generación del set de datos para Café Selecto
random.seed(42)  # Para resultados reproducibles

# Listas base para los datos
regions = ["Norte", "Sur", "Este", "Oeste"]
product_categories = ["Café", "Tés", "Repostería", "Snacks Salados"]
store_sizes = {"Norte": ["mediana", "grande"], "Sur": ["pequeña", "mediana"], "Este": ["mediana", "grande"], "Oeste": ["pequeña", "mediana"]}
payment_methods = ["Efectivo", "Tarjeta", "Digital"]
customer_types = ["Nuevo", "Recurrente"]
shifts = ["mañana", "tarde", "noche"]

# Generación de coordenadas y tiendas por región
store_catalog = []
store_id_counter = 1
for region, (lat_base, lon_base) in {
    "Norte": (25.6866, -100.3161),
    "Sur": (17.9895, -92.9475),
    "Este": (19.4326, -99.1332),
    "Oeste": (20.6597, -103.3496)
}.items():
    for _ in range(random.randint(10, 15)):  # Generar entre 10 y 15 tiendas por región
        store_catalog.append({
            "store_id": f"S-{store_id_counter}",
            "region": region,
            "latitude": round(lat_base + random.uniform(-0.4, 0.4), 6),
            "longitude": round(lon_base + random.uniform(-0.4, 0.4), 6)
        })
        store_id_counter += 1

# Parámetros para las fechas
start_date = datetime(2024, 1, 1)
n_samples = 5000

# Relación entre fechas y clima
def get_weather(date):
    if date.month in [12, 1, 2]:
        return random.choice(["frío", "lluvioso"])
    elif date.month in [3, 4, 5]:
        return random.choice(["soleado", "lluvioso"])
    elif date.month in [6, 7, 8]:
        return "soleado"
    else:
        return random.choice(["soleado", "lluvioso"])

data = {
    "transaction_id": [],
    "ticket_id": [],
    "region": [],
    "store_id": [],
    "latitude": [],
    "longitude": [],
    "date": [],
    "transaction_time": [],
    "shift": [],
    "day_of_week": [],
    "season": [],
    "sales": [],
    "product_category": [],
    "product_sku": [],
    "quantity": [],
    "promotion": [],
    "weather": [],
    "store_size": [],
    "staff_count": [],
    "customer_type": [],
    "payment_method": []
}

# Rangos de tiempo por turno
def get_time_for_shift(shift):
    if shift == "mañana":
        return (datetime.min + timedelta(hours=random.randint(6, 11), minutes=random.randint(0, 59))).strftime("%H:%M:%S")
    elif shift == "tarde":
        return (datetime.min + timedelta(hours=random.randint(12, 17), minutes=random.randint(0, 59))).strftime("%H:%M:%S")
    elif shift == "noche":
        return (datetime.min + timedelta(hours=random.randint(18, 21), minutes=random.randint(0, 59))).strftime("%H:%M:%S")

# Pesos de categorías por región
regional_preferences = {
    "Norte": {"Café": 0.5, "Tés": 0.1, "Repostería": 0.2, "Snacks Salados": 0.2},
    "Sur": {"Café": 0.3, "Tés": 0.4, "Repostería": 0.2, "Snacks Salados": 0.1},
    "Este": {"Café": 0.4, "Tés": 0.2, "Repostería": 0.3, "Snacks Salados": 0.1},
    "Oeste": {"Café": 0.5, "Tés": 0.2, "Repostería": 0.3, "Snacks Salados": 0.0}
}

# Generación del dataset
current_ticket_id = None
transactions_in_ticket = 0
current_store = None
current_payment_method = None
current_customer_type = None
current_transaction_time = None

for i in range(n_samples):
    # Crear un nuevo ticket si es el inicio o después de 3 transacciones
    if transactions_in_ticket == 0 or transactions_in_ticket >= random.randint(1, 3):
        current_ticket_id = f"T-{random.randint(1000, 9999)}"
        current_store = random.choice(store_catalog)
        current_payment_method = random.choice(payment_methods)
        current_customer_type = random.choice(customer_types)
        transactions_in_ticket = 0

    transactions_in_ticket += 1

    # Datos del ticket actual
    region = current_store["region"]
    store_id = current_store["store_id"]
    latitude = current_store["latitude"]
    longitude = current_store["longitude"]

    date = start_date + timedelta(days=random.randint(0, 364))  # Distribuir en todo el año 2024
    shift = random.choices(shifts, weights=[0.4, 0.4, 0.2])[0]  # Mayor peso en mañana y tarde
    current_transaction_time = get_time_for_shift(shift)

    day_of_week = date.strftime("%A").lower()
    season = "invierno" if date.month in [1, 2, 12] else "primavera" if date.month in [3, 4, 5] else "verano" if date.month in [6, 7, 8] else "otoño"
    weather = get_weather(date)

    # Diferenciación de ventas por categoría
    category_weights = regional_preferences[region]
    product_category = random.choices(list(category_weights.keys()), weights=category_weights.values())[0]

    # SKU específico y cantidad vendida
    product_skus = {
        "Café": ["Latte", "Espresso", "Americano"],
        "Tés": ["Té Verde", "Té Chai", "Té Negro"],
        "Repostería": ["Muffin", "Croissant", "Donut"],
        "Snacks Salados": ["Chips", "Pretzels", "Palomitas"]
    }
    product_sku = random.choice(product_skus[product_category])
    quantity = random.randint(1, 5)  # Cantidad de productos vendidos

    # Impacto de promociones
    promotion = random.random() < 0.1 if product_category in ["Café", "Repostería"] else random.random() < 0.05
    if promotion:
        quantity += 1  # Incremento por promoción

    # Precio promedio por producto
    price_per_unit = random.uniform(5, 15) if product_category in ["Café", "Tés"] else random.uniform(10, 25)
    sales = round(price_per_unit * quantity, 2)

    # Agregar datos al diccionario
    data["transaction_id"].append(i + 1)
    data["ticket_id"].append(current_ticket_id)
    data["region"].append(region)
    data["store_id"].append(store_id)
    data["latitude"].append(latitude)
    data["longitude"].append(longitude)
    data["date"].append(date.strftime("%Y-%m-%d"))
    data["transaction_time"].append(current_transaction_time)
    data["shift"].append(shift)
    data["day_of_week"].append(day_of_week)
    data["season"].append(season)
    data["sales"].append(sales)
    data["product_category"].append(product_category)
    data["product_sku"].append(product_sku)
    data["quantity"].append(quantity)
    data["promotion"].append("Sí" if promotion else "No")
    data["weather"].append(weather)
    data["store_size"].append(random.choice(store_sizes[region]))
    data["staff_count"].append(random.randint(5, 15))
    data["customer_type"].append(current_customer_type)
    data["payment_method"].append(current_payment_method)

# Crear DataFrame
df = pd.DataFrame(data)

# Guardar el dataset en un archivo CSV
df.to_csv("data/ventas_cafe_selecto.csv", index=False)

print("Dataset generado y guardado como 'ventas_cafe_selecto.csv' con estructura completa y datos enriquecidos")


Dataset generado y guardado como 'ventas_cafe_selecto.csv' con estructura completa y datos enriquecidos


In [2]:
df.head()

,transaction_id,ticket_id,region,store_id,latitude,longitude,date,transaction_time,shift,day_of_week,...,sales,product_category,product_sku,quantity,promotion,weather,store_size,staff_count,customer_type,payment_method
0,1,T-8668,Este,S-34,19.676637,-99.212268,2024-12-14,17:56:00,tarde,saturday,...,12.37,Repostería,Croissant,1,No,lluvioso,mediana,9,Nuevo,Tarjeta
1,2,T-8668,Este,S-34,19.676637,-99.212268,2024-04-01,12:55:00,tarde,monday,...,78.04,Repostería,Donut,5,No,lluvioso,mediana,13,Nuevo,Tarjeta
2,3,T-8668,Este,S-34,19.676637,-99.212268,2024-01-01,15:01:00,tarde,monday,...,46.34,Snacks Salados,Pretzels,2,No,frío,mediana,6,Nuevo,Tarjeta
3,4,T-8962,Norte,S-5,25.312026,-100.641144,2024-03-06,16:10:00,tarde,wednesday,...,57.16,Tés,Té Negro,4,No,lluvioso,mediana,9,Nuevo,Digital
4,5,T-8962,Norte,S-5,25.312026,-100.641144,2024-12-09,15:57:00,tarde,monday,...,10.88,Café,Latte,1,No,lluvioso,mediana,14,Nuevo,Digital
